In [ ]:
# Run this first to clone the published DSA implementation.
from pathlib import Path
import subprocess
import sys
import os
import json
import hashlib
import shutil

REPO_URL = "https://github.com/sttruji/dynamic-state-anchoring.git"
REPO_REF = "main"  # Set a commit SHA here to reproduce a particular implementation.
REPO_DIR = Path("/content/dynamic-state-anchoring")

def git_output(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()

if not REPO_DIR.exists():
    try:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Check that the repository is reachable, then rerun. "
                           "Private repositories need authenticated Git access configured separately.") from exc
else:
    remote = git_output("remote", "get-url", "origin").rstrip("/").removesuffix(".git")
    if remote != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("Existing checkout has a different remote; choose another REPO_DIR.")
    if git_output("status", "--porcelain"):
        raise RuntimeError("Existing checkout has local changes. Preserve them before updating it.")
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
REPO_COMMIT = git_output("rev-parse", "HEAD")
if not (REPO_DIR / "requirements-colab.txt").is_file():
    raise RuntimeError("This ref does not contain the DSA implementation.")
print("DSA implementation:", REPO_COMMIT)


# DSA saved-checkpoint diagnostics

Choose a **T4 GPU** and run these cells in order. This notebook targets the
completed `gpt2-pilot-v1-ba4e69315086` experiment, independently of the code commit
fetched above. It never calls the trainer, changes checkpoint weights, downloads
the corpus or selects settings using validation/test answers.

It checks all 32 training questions using question-only generation. Small probes
then vary loop depth and anchor writes on the first four prepared training rows.
Gradients are measured on the first two prepared training rows at K=1,2,4,8,16;
at the original K=16 it also separates weighted answer/CoT/gate/alignment losses.
No optimizer updates are performed. The latent vectors are measured directly;
generated text is an answer-quality check, not a transcript of latent reasoning.


In [ ]:
# Reuse Colab's CUDA/PyTorch stack; isolate project packages from the notebook kernel.
# The helper uses --without-pip --system-site-packages and targets this environment explicitly.
VENV_DIR = Path("/content/dsa-venv")
subprocess.run([sys.executable, str(REPO_DIR / "scripts/setup_colab.py"),
                "--venv-dir", str(VENV_DIR),
                "--requirements", str(REPO_DIR / "requirements-colab.txt")], check=True)
PYTHON = str(VENV_DIR / "bin/python")
print("Project Python:", PYTHON)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/dynamic-state-anchoring")
for name in ("corpus", "prepared", "runs", "reports", "cache/huggingface"):
    (PROJECT_DIR / name).mkdir(parents=True, exist_ok=True)
# Set before any subprocess imports Hugging Face packages.
os.environ["HF_HOME"] = str(PROJECT_DIR / "cache/huggingface")
os.environ["HF_HUB_CACHE"] = str(PROJECT_DIR / "cache/huggingface/hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TRANSFORMERS_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
print("Persistent storage:", PROJECT_DIR)


In [ ]:
# Keep the original experiment selected when fetching newer diagnostic code.
RUN_NAME = "gpt2-pilot-v1-ba4e69315086"
EXPECTED_STEP = 200
ARMS = ["static", "learned"]
DEVICE = "cuda"
GRADIENT_LIMIT = 2
PROBE_LIMIT = 4
DEPTH_GRID = [1, 2, 4, 8, 16]
MAX_NEW_TOKENS = 64
DIAGNOSTIC_NAME = "checkpoint-diagnostics-v1"  # Change if diagnostic settings/code change.

RUN_ROOT = PROJECT_DIR / "runs" / RUN_NAME
REPORT_ROOT = PROJECT_DIR / "reports" / RUN_NAME / DIAGNOSTIC_NAME
assert Path(RUN_NAME).name == RUN_NAME and Path(DIAGNOSTIC_NAME).name == DIAGNOSTIC_NAME
manifests = {arm: json.loads((RUN_ROOT / arm / "run_manifest.json").read_text()) for arm in ARMS}
for arm, manifest in manifests.items():
    latest = json.loads((RUN_ROOT / arm / "latest.json").read_text())
    if manifest["completed_steps"] != EXPECTED_STEP or latest["step"] != EXPECTED_STEP:
        raise RuntimeError(f"{arm}: expected saved step {EXPECTED_STEP}; inspect the run before proceeding.")
expected_hashes = {manifest["contract"]["prepared_manifest_sha256"] for manifest in manifests.values()}
assert len(expected_hashes) == 1, "Arms refer to different prepared data."
expected_hash = next(iter(expected_hashes))
candidates = sorted(p.parent for p in (PROJECT_DIR / "prepared").glob("*/manifest.json")
                    if hashlib.sha256(p.read_bytes()).hexdigest() == expected_hash)
if not candidates:
    raise FileNotFoundError("The original prepared cache is missing from Drive. Restore it before diagnostics.")
PREPARED_DRIVE = candidates[0]
PREPARED_LOCAL = Path("/content/dsa-diagnostic-prepared") / PREPARED_DRIVE.name
shutil.copytree(PREPARED_DRIVE, PREPARED_LOCAL, dirs_exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
import runpy
_run_logged = runpy.run_path(str(REPO_DIR / "scripts/colab_process.py"))["run_logged"]
def run_project(*args):
    return _run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR,
                       env=os.environ.copy(), log_dir=REPORT_ROOT / "logs")
print("Diagnostic code:", REPO_COMMIT)
print("Original training run:", RUN_ROOT)
print("Original prepared cache:", PREPARED_DRIVE)
print("Diagnostic reports:", REPORT_ROOT)


In [ ]:
run_project("-c", "import torch; print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available()); "
            "assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, reconnect and rerun setup.'")
run_project("-m", "unittest", "discover", "-s", "tests", "-p", "test_pilot_diagnostics.py", "-q")
print("GPU access and diagnostic correctness checks passed.")


The next cell processes one arm at a time and validates its saved model/data
identities. Each completed prediction and gradient case is saved to Drive;
rerunning with identical settings resumes unfinished work. Each component of a
joint gradient probe shares the same forward graph and dropout realization.
Short-depth gradient probes use answer loss only, keeping the original training
labels intact. Weight checksums are read before execution; this can take time on
Drive before the first prediction appears.

The full-objective probe uses training mode, dropout and straight-through gates.
Inference probes use hard gates. Norms are measured **before clipping**, and the
script verifies the model state is unchanged afterward. State/input gradients
come from actual tensors on the loss path. Absent gradients are saved as null,
separately from zero gradients. A nonfinite norm is also saved as null with the
parameter nonfinite count where applicable.


In [ ]:
diagnostic_reports = {}
for arm in ARMS:
    print(f"Diagnosing {arm}; saved step {EXPECTED_STEP}; no training updates", flush=True)
    out = REPORT_ROOT / arm
    run_project("scripts/diagnose_dsa_pilot.py", "--prepared-dir", PREPARED_LOCAL,
                "--run-dir", RUN_ROOT / arm, "--output-dir", out, "--device", DEVICE,
                "--gradient-limit", GRADIENT_LIMIT, "--probe-limit", PROBE_LIMIT,
                "--depth-grid", *DEPTH_GRID, "--max-new-tokens", MAX_NEW_TOKENS)
    diagnostic_reports[arm] = json.loads((out / "report.json").read_text())
print("Both arms complete. Continue to the summary and export cells.")


In [ ]:
def scientific(value):
    return "absent/nonfinite" if value is None else f"{value:.3e}"

for arm, report in diagnostic_reports.items():
    print(f"\n{arm}: optimizer updates={report['optimizer_updates']}; weights unchanged={report['model_state_unchanged']}")
    print(f"{'condition':32} {'n':>3} {'correct':>8} {'accuracy':>9} {'writes':>9}")
    for name, result in report["prediction_reports"].items():
        print(f"{name:32} {result['total']:3d} {result['correct']:8d} {result['accuracy']:9.2%} {result['mean_write_rate']:9.2%}")
        if "paired_with_normal_same_questions" in result:
            print("  Paired changes:", result["paired_with_normal_same_questions"])
    print("Training-label gate metrics:", report["prediction_reports"]["normal_training_set"].get("controller_training_labels"))
    print("\nGradient norms before clipping (training-mode probes):")
    for probe in report["gradient_reports"]:
        for loss_name, loss in probe["losses"].items():
            grads = loss["parameter_gradients"]
            first, last = loss["states"][0], loss["states"][-1]
            print(f"{probe['path']} {loss_name:6}: parameters={scientific(grads['global_norm_before_clipping'])}; "
                  f"first workspace input={scientific(first['workspace_input_gradient_l2'])}; "
                  f"last workspace input={scientific(last['workspace_input_gradient_l2'])}; "
                  f"nonfinite parameter gradients={grads['nonfinite_parameter_gradients']}")
    print("Full report:", REPORT_ROOT / arm / "report.json")
summary = {"run_name": RUN_NAME, "diagnostic_code_commit": REPO_COMMIT,
           "purpose": "training-only diagnostics; no optimizer updates or benchmark claim",
           "reports": {arm: str(REPORT_ROOT / arm / "report.json") for arm in ARMS}}
(REPORT_ROOT / "notebook-summary.json").write_text(json.dumps(summary, indent=2) + "\n")


Read `normal_training_set` first: it evaluates all 32 training questions. High
training accuracy with poor validation suggests a generalization gap; low
training accuracy leaves optimization, undertraining and decoding as candidates.
The intervention conditions use only four questions, so their percentages are
mechanism probes rather than comparative research results. The ordinary K=16
predictions for those same four questions are the first four records in
`normal-training-set/predictions.jsonl`.

Compare answer-only gradient norms across depths on each identical example.
Then inspect the weighted loss contributions at K=16. Norms of different loss
gradients do not add like scalars: directions and cancellation also matter.
Small or changing state vectors do not by themselves establish useful reasoning;
large gradients need interpretation alongside the actual answer behavior.
The full state traces, generated token IDs, EOS/cap stop reasons and checkpoint
identities are saved for review. No validation or test answers are used here.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

archive = Path("/content/dsa-checkpoint-diagnostics.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
    for path in sorted(REPORT_ROOT.rglob("*")):
        if path.is_file() and path.suffix in {".json", ".jsonl", ".log"}:
            bundle.write(path, path.relative_to(REPORT_ROOT))
print("Diagnostic export:", archive)
files.download(str(archive))
